In [5]:
"""
OralVision-M4 — Classifier-Family Comparison on Frozen CoordAtt Embeddings
(REBUILT ON THE CLEAN, DEDUPLICATED DATASET)
RTX 2050 HARDENED VERSION — safe for 4GB laptop VRAM
=================================================================================
PURPOSE
-------
Same as the original script (paper Section 4.7 classifier-family comparison:
Logistic Regression vs. SVM vs. Random Forest vs. XGBoost on frozen CoordAtt
embeddings), rebuilt on oral_dataset_FINAL_CLEAN — but hardened so it runs
reliably on a 4GB RTX 2050 laptop GPU instead of assuming desktop-class VRAM.

WHAT'S DIFFERENT FROM THE ORIGINAL SCRIPT
------------------------------------------
1. Smaller default batch size (16 instead of 32) — RTX 2050 has only 4GB VRAM
   and EfficientNet-B0 + CoordAtt feature maps can spike memory on 224x224
   batches of 32, especially on Windows where the CUDA context itself eats
   ~300-500MB before you load anything.
2. Automatic OOM recovery: embedding extraction wraps each batch in a
   try/except for torch.cuda.OutOfMemoryError and halves the effective batch
   (by re-running that chunk split in two) instead of crashing the whole run.
3. Mixed precision (torch.autocast) during embedding extraction — inference
   only, no gradients — roughly halves activation memory with no accuracy
   impact since the backbone is frozen and we're not backpropping.
4. torch.cuda.empty_cache() called between extraction and classifier fitting,
   and again between each sklearn/xgboost classifier fit, since PyTorch's
   allocator won't release memory back to the OS on its own and RF/XGBoost
   fitting on CPU doesn't need the GPU memory but the process still holds it.
5. Model load has a CPU fallback: if CUDA OOMs just moving the model onto the
   device (rare, but happens if something else is already using the GPU —
   e.g. Chrome hardware acceleration, another training run left running),
   the script logs a warning and continues on CPU rather than crashing.
6. pin_memory=True on the DataLoaders when CUDA is available (safe no-op on
   CPU) for faster host->device transfer.
7. cudnn.benchmark is left OFF (default) rather than turned on — benchmark
   mode searches multiple conv algorithms and picks the fastest, but on a
   4GB card the memory used *during that search* can itself trigger an OOM
   that wouldn't happen with benchmark off. Not worth the speedup here since
   this is a one-shot inference pass, not a long training loop.
8. VRAM logging: prints total/allocated/reserved GPU memory before and after
   embedding extraction so you can see headroom instead of finding out from
   a crash.
9. Everything else (model architecture, classifiers, metrics, report
   format) is unchanged — only how the GPU is used and how the dataset is
   located/labeled differ, so the numbers stay comparable to a run of the
   original script.

FIX (this revision): dataset discovery now mirrors
m4_coordatt_logreg_CLEAN_DATASET.ipynb exactly instead of assuming a fixed
layout:
- train/val/test directories are found by recursive NAME search
  (find_split_directory), not by joining DATASET_ROOT with a fixed "train"/
  "test" path — this is what let the earlier version silently find 0 images
  if the folders were nested differently than assumed.
- CLASS_NAMES is no longer hardcoded. Class names are discovered from the
  actual subfolders under train/val/test (discover_classes), and it's a hard
  error if train/val/test disagree on what those classes are — exactly the
  same contract the training notebook enforces.
- Valid image extensions are now .jpg/.jpeg/.png/.bmp/.webp (matching the
  training notebook) instead of just .jpg/.jpeg/.png.
- A val/validation directory is required to exist even though this script
  only reads train/test, because discover_classes needs all three splits
  to cross-check class names against each other, same as training does.

>>> EDIT THE CONFIG BLOCK BELOW BEFORE RUNNING <<<
"""

import os
import csv
import gc
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


# =============================================================================
# CONFIG — edit these paths for your machine
# =============================================================================

CHECKPOINT_PATH = r"C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED\best_model.pth"
DATASET_ROOT = Path(r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN")
OUTPUT_DIR = r"C:\Users\R Y Z E N\Desktop\m4_classifier_comparison_CLEAN"

IMG_SIZE = 224

# RTX 2050 (4GB VRAM) — 16 is a safe default. If you still hit OOM, drop to 8.
# The OOM-recovery logic below will auto-split a batch that fails, but
# starting smaller means fewer retries and a faster overall run.
BATCH_SIZE = 16

# Windows + CUDA: keep this at 0. Multi-worker DataLoader on Windows spawns
# new processes that each re-import CUDA context, which wastes VRAM and can
# itself cause OOM on a 4GB card. Not a speed knob worth touching here.
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Use mixed precision during frozen-embedding extraction. Safe: no training,
# no gradients, backbone weights untouched — this only reduces activation
# memory during the forward pass.
USE_AMP = DEVICE == "cuda"

# NOTE: CLASS_NAMES is no longer hardcoded here. The training notebook
# (m4_coordatt_logreg_CLEAN_DATASET.ipynb) discovers class names dynamically
# from whatever folders actually exist under train/val/test, and finds the
# train/val/test directories themselves by recursive name search rather than
# assuming a fixed layout. A hardcoded list here was the actual bug that
# produced an empty test set — this script now mirrors that exact discovery
# logic so it can never silently mismatch the real folder names/paths again.
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# =============================================================================
# MODEL DEFINITION — must match the checkpoint's actual trained architecture
# exactly (conv1 bias=False; single Linear(1280 -> 6) head, no dropout).
# =============================================================================
class CoordAtt(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, channels // reduction)
        self.conv1 = nn.Conv2d(channels, mip, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish(inplace=True)
        self.conv_h = nn.Conv2d(mip, channels, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, channels, kernel_size=1, stride=1, padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        _, _, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.sigmoid(self.conv_h(x_h))
        a_w = self.sigmoid(self.conv_w(x_w))
        return identity * a_h * a_w


class EfficientNetCoordAtt(nn.Module):
    def __init__(self, num_classes, ca_reduction=32):
        super().__init__()
        base = models.efficientnet_b0(weights=None)
        self.features = base.features
        self.coordatt = CoordAtt(1280, reduction=ca_reduction)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1280, num_classes)

    def embed(self, x):
        """Returns the 1280-d frozen embedding (post-CoordAtt, pre-classifier)."""
        x = self.features(x)
        x = self.coordatt(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        return self.classifier(self.embed(x))


# =============================================================================
# DIRECTORY / CLASS DISCOVERY — identical logic to the training notebook,
# so this script can never disagree with it about where the data lives or
# what the classes are called.
# =============================================================================
def find_split_directory(root, names):
    """Find train/val/test directory recursively (matches the training
    notebook exactly: searches by folder NAME, not by a fixed path, and
    picks the directory with the most class subfolders if there are
    multiple candidates)."""
    if isinstance(names, str):
        names = [names]
    lname_set = {x.lower() for x in names}

    candidates = []
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        if p.name.lower() in lname_set:
            class_dirs = [x for x in p.iterdir() if x.is_dir()]
            if len(class_dirs) >= 2:
                candidates.append((len(class_dirs), p))

    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


def discover_classes(*split_dirs):
    """Class names come from the actual folders on disk, and every split
    must agree — exactly as in the training notebook. This is what a
    hardcoded CLASS_NAMES list can never guarantee."""
    class_sets = [{p.name for p in d.iterdir() if p.is_dir()} for d in split_dirs]
    first = class_sets[0]
    if any(cs != first for cs in class_sets[1:]):
        lines = ["Class directories do not match across splits:"]
        for d, cs in zip(split_dirs, class_sets):
            lines.append(f"  {d}: {sorted(cs)}")
        raise RuntimeError("\n".join(lines))
    return sorted(first)


def collect_samples(split_dir, class_names):
    samples = []
    for class_name in class_names:
        class_dir = split_dir / class_name
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                samples.append((str(path), class_names.index(class_name)))
    return samples


# =============================================================================
# DATASET
# =============================================================================
class FolderDataset(Dataset):
    def __init__(self, samples, transform):
        self.transform = transform
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            img = img.convert("RGB")
        return self.transform(img), label


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


def log_vram(tag):
    if DEVICE != "cuda":
        return
    total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)
    reserved = torch.cuda.memory_reserved(0) / (1024 ** 2)
    log(f"  [VRAM {tag}] total={total:.0f}MB allocated={allocated:.0f}MB reserved={reserved:.0f}MB")


def load_model_safely(num_classes, checkpoint_path):
    """Loads the model and moves it to DEVICE, falling back to CPU if the
    GPU is already under memory pressure from something else."""
    model = EfficientNetCoordAtt(num_classes=num_classes)
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state_dict = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)) if isinstance(ckpt, dict) else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        log(f"  [WARN] missing={missing} unexpected={unexpected}")
        log("  --> Do NOT trust downstream numbers until this is empty. Stop and fix the architecture match first.")
    else:
        log("  Checkpoint loaded cleanly — no missing/unexpected keys.")

    global DEVICE
    if DEVICE == "cuda":
        try:
            model.to(DEVICE)
        except torch.cuda.OutOfMemoryError:
            log("  [WARN] CUDA OOM while moving model to GPU — falling back to CPU for this run.")
            log("  (Check for other processes holding VRAM: other Python/CUDA jobs, browser hardware accel, etc.)")
            torch.cuda.empty_cache()
            DEVICE = "cpu"
            model.to(DEVICE)
    else:
        model.to(DEVICE)
    return model


# =============================================================================
# EMBEDDING EXTRACTION — with OOM-safe batch splitting and AMP
# =============================================================================
def _embed_batch(model, imgs):
    """Runs model.embed on a single batch, using autocast when enabled."""
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            emb = model.embed(imgs)
    else:
        emb = model.embed(imgs)
    return emb.float()


def _embed_with_oom_retry(model, imgs):
    """Tries to embed a batch; on CUDA OOM, splits it in half and retries
    recursively until it fits (down to single images)."""
    try:
        return _embed_batch(model, imgs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        n = imgs.size(0)
        if n == 1:
            raise  # can't split further — genuinely out of memory
        log(f"    [OOM] batch of {n} didn't fit — splitting and retrying")
        mid = n // 2
        first = _embed_with_oom_retry(model, imgs[:mid])
        second = _embed_with_oom_retry(model, imgs[mid:])
        return torch.cat([first, second], dim=0)


def extract_embeddings(model, loader, tag):
    log(f"Extracting embeddings: {tag} ({len(loader.dataset)} images)")
    log_vram(f"before {tag}")
    feats, labels = [], []
    model.eval()
    with torch.no_grad():
        for imgs, y in loader:
            imgs = imgs.to(DEVICE, non_blocking=(DEVICE == "cuda"))
            emb = _embed_with_oom_retry(model, imgs)
            feats.append(emb.cpu().numpy())
            labels.append(y.numpy())
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    log_vram(f"after {tag}")
    return np.concatenate(feats), np.concatenate(labels)


def free_gpu_memory():
    """Call between classifier fits — RF/XGBoost run on CPU but the process
    still holds whatever CUDA reserved during embedding extraction unless
    explicitly released."""
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# =============================================================================
# MAIN
# =============================================================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    log(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(0)}")
        log(f"  Mixed precision (AMP): {USE_AMP}")
        log(f"  Batch size: {BATCH_SIZE} (auto-splits further on OOM)")

    eval_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"\nDataset directory not found:\n{DATASET_ROOT}\n"
            "Check DATASET_ROOT at the top of the script."
        )

    # Same discovery logic as the training notebook: find the actual
    # train/val/test directories by name (recursively — doesn't assume a
    # flat layout), then discover class names from what's really on disk.
    # A val/validation directory is required to exist (the training script
    # needs it), even though this comparison script itself only uses
    # train/test.
    train_dir = find_split_directory(DATASET_ROOT, "train")
    val_dir = find_split_directory(DATASET_ROOT, ["val", "validation"])
    test_dir = find_split_directory(DATASET_ROOT, "test")

    if train_dir is None:
        raise FileNotFoundError(f"Could not find a TRAIN directory under:\n{DATASET_ROOT}")
    if val_dir is None:
        raise FileNotFoundError(f"Could not find a VAL/VALIDATION directory under:\n{DATASET_ROOT}")
    if test_dir is None:
        raise FileNotFoundError(f"Could not find a TEST directory under:\n{DATASET_ROOT}")

    log(f"Train directory: {train_dir}")
    log(f"Val directory  : {val_dir}")
    log(f"Test directory : {test_dir}")

    class_names = discover_classes(train_dir, val_dir, test_dir)
    log(f"Classes ({len(class_names)}): {class_names}")

    train_samples = collect_samples(train_dir, class_names)
    test_samples = collect_samples(test_dir, class_names)

    if not train_samples:
        raise RuntimeError(f"Train split loaded 0 images from: {train_dir}")
    if not test_samples:
        raise RuntimeError(f"Test split loaded 0 images from: {test_dir}")

    log(f"Train images: {len(train_samples)}  |  Test images: {len(test_samples)}")

    train_ds = FolderDataset(train_samples, eval_transform)
    test_ds = FolderDataset(test_samples, eval_transform)

    pin_memory = DEVICE == "cuda"
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=pin_memory)

    log(f"Loading checkpoint: {CHECKPOINT_PATH}")
    model = load_model_safely(len(class_names), CHECKPOINT_PATH)

    # Reference: the model's own jointly-trained linear head.
    joint_preds, joint_labels = [], []
    model.eval()
    with torch.no_grad():
        for imgs, y in test_loader:
            imgs = imgs.to(DEVICE, non_blocking=(DEVICE == "cuda"))
            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(imgs)
            else:
                logits = model(imgs)
            joint_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            joint_labels.append(y.numpy())
    joint_preds = np.concatenate(joint_preds)
    joint_labels = np.concatenate(joint_labels)
    joint_report = classification_report(joint_labels, joint_preds, target_names=class_names,
                                          output_dict=True, zero_division=0)
    free_gpu_memory()

    X_train, y_train = extract_embeddings(model, train_loader, "train")
    X_test, y_test = extract_embeddings(model, test_loader, "test")

    # Backbone is no longer needed once embeddings are extracted — release
    # the whole model from GPU memory before fitting CPU-side classifiers.
    if DEVICE == "cuda":
        model.to("cpu")
        free_gpu_memory()
        log_vram("after releasing backbone")

    classifiers = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
        "SVM (RBF)": SVC(kernel="rbf", random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    }
    if HAS_XGB:
        classifiers["XGBoost"] = XGBClassifier(
            n_estimators=300, use_label_encoder=False, eval_metric="mlogloss",
            random_state=SEED, n_jobs=-1,
        )
    else:
        log("  [WARN] xgboost not installed — skipping. Run: pip install xgboost --break-system-packages")

    results = []
    for name, clf in classifiers.items():
        log(f"Fitting: {name}")
        t0 = time.perf_counter()
        clf.fit(X_train, y_train)
        t_fit = time.perf_counter() - t0

        t0 = time.perf_counter()
        preds = clf.predict(X_test)
        t_infer = time.perf_counter() - t0

        report = classification_report(y_test, preds, target_names=class_names,
                                        output_dict=True, zero_division=0)
        results.append({
            "classifier": name,
            "accuracy": report["accuracy"],
            "macro_precision": report["macro avg"]["precision"],
            "macro_recall": report["macro avg"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"],
            "runtime_sec": t_fit + t_infer,
        })
        log(f"  {name}: acc={report['accuracy']:.4f} macro_f1={report['macro avg']['f1-score']:.4f} "
            f"runtime={t_fit + t_infer:.2f}s")
        free_gpu_memory()

    results.append({
        "classifier": "Linear (joint end-to-end head)",
        "accuracy": joint_report["accuracy"],
        "macro_precision": joint_report["macro avg"]["precision"],
        "macro_recall": joint_report["macro avg"]["recall"],
        "macro_f1": joint_report["macro avg"]["f1-score"],
        "runtime_sec": None,
    })

    # ---- write report ----
    txt_path = os.path.join(OUTPUT_DIR, "classifier_family_comparison_report.txt")
    with open(txt_path, "w") as f:
        f.write("OralVision-M4 — Classifier-Family Comparison (Clean Dataset, RTX 2050 Hardened Run)\n")
        f.write("=" * 85 + "\n")
        f.write(f"Checkpoint: {CHECKPOINT_PATH}\n")
        f.write(f"Device used: {DEVICE}\n")
        f.write(f"Train embeddings: {X_train.shape[0]} x {X_train.shape[1]}\n")
        f.write(f"Test embeddings:  {X_test.shape[0]} x {X_test.shape[1]}\n\n")
        f.write(f"{'Classifier':<32}{'Accuracy':>10}{'Precision':>12}{'Recall':>10}{'F1':>10}{'Runtime(s)':>12}\n")
        for r in results:
            rt = f"{r['runtime_sec']:.2f}" if r["runtime_sec"] is not None else "N/A*"
            f.write(f"{r['classifier']:<32}{r['accuracy']:>10.4f}{r['macro_precision']:>12.4f}"
                    f"{r['macro_recall']:>10.4f}{r['macro_f1']:>10.4f}{rt:>12}\n")
        f.write("\n*Not separately measured: this is the backbone's own head, evaluated as part\n"
                "of the standard forward pass rather than fit as an independent classifier.\n")

    csv_path = os.path.join(OUTPUT_DIR, "classifier_family_comparison.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["classifier", "accuracy", "macro_precision",
                                                "macro_recall", "macro_f1", "runtime_sec"])
        writer.writeheader()
        writer.writerows(results)

    log(f"Wrote {txt_path}")
    log(f"Wrote {csv_path}")
    if not HAS_XGB:
        log("Re-run after installing xgboost to fill in that row before this goes in the paper.")


if __name__ == "__main__":
    main()

[05:15:40] Device: cuda
[05:15:40]   GPU: NVIDIA GeForce RTX 2050
[05:15:40]   Mixed precision (AMP): True
[05:15:40]   Batch size: 16 (auto-splits further on OOM)
[05:15:54] Train directory: C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\train
[05:15:54] Val directory  : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\val
[05:15:54] Test directory : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\test
[05:15:54] Classes (6): ['Calculus', 'Caries', 'Gingivitis', 'Hypodontia', 'Mouth_Ulcer', 'Tooth_Discoloration']
[05:15:57] Train images: 17700  |  Test images: 1265
[05:15:57] Loading checkpoint: C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED\best_model.pth


C:\Users\R Y Z E N\AppData\Local\Temp\ipykernel_4936\2153528047.py:275: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


[05:15:57]   Checkpoint loaded cleanly — no missing/unexpected keys.
[05:16:07] Extracting embeddings: train (17700 images)
[05:16:07]   [VRAM before train] total=4096MB allocated=41MB reserved=62MB
[05:17:52]   [VRAM after train] total=4096MB allocated=43MB reserved=62MB
[05:17:52] Extracting embeddings: test (1265 images)
[05:17:52]   [VRAM before test] total=4096MB allocated=41MB reserved=62MB
[05:17:59]   [VRAM after test] total=4096MB allocated=42MB reserved=62MB
[05:17:59]   [VRAM after releasing backbone] total=4096MB allocated=25MB reserved=56MB
[05:17:59] Fitting: Logistic Regression
[05:18:00]   Logistic Regression: acc=0.9937 macro_f1=0.9771 runtime=0.88s
[05:18:00] Fitting: SVM (RBF)
[05:18:03]   SVM (RBF): acc=0.9905 macro_f1=0.9663 runtime=3.50s
[05:18:04] Fitting: Random Forest
[05:18:12]   Random Forest: acc=0.9929 macro_f1=0.9737 runtime=8.38s
[05:18:12] Fitting: XGBoost


C:\Users\R Y Z E N\shopgenie\lib\site-packages\xgboost\training.py:183: UserWarning: [05:18:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[05:19:42]   XGBoost: acc=0.9897 macro_f1=0.9613 runtime=89.30s
[05:19:42] Wrote C:\Users\R Y Z E N\Desktop\m4_classifier_comparison_CLEAN\classifier_family_comparison_report.txt
[05:19:42] Wrote C:\Users\R Y Z E N\Desktop\m4_classifier_comparison_CLEAN\classifier_family_comparison.csv
